In [ ]:
%matplotlib inline

# NOTE: %reset wipes every variable in the current kernel session, and normally
# prompts "Proceed (y/[n])?" interactively when run in a live notebook - kept
# here to ensure a clean run, but will hang if this notebook is ever run
# non-interactively (e.g. via jupyter nbconvert --execute).
%reset

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import nibabel as nib
import plotly.graph_objects as go
import seaborn as sns
import statsmodels.api as sm

from pathlib import Path
from scipy import stats
from scipy.integrate import simpson
from scipy.io import loadmat
from scipy.spatial import cKDTree
from scipy.stats import pearsonr
from skfda.representation.basis import BSplineBasis
from statsmodels.formula.api import ols
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import sys
import warnings

warnings.filterwarnings('ignore')

# Root of this project (the folder containing this notebook, or an ancestor of
# it, named 'FDA'). Walks up from the current working directory until it finds
# one. (Same pattern used throughout this project's notebooks.)
projectRoot = Path().resolve().parent

sys.path.append(str(projectRoot / 'functions'))
import fNIRSData

# ---- USER SETTINGS ----
# Root of the drive/mount holding the MRI/mesh data (the same `driveName` used
# throughout the imageReconstruction MATLAB repo). Edit for your machine —
# pathlib.Path handles the rest cross-platform, no OS-specific branching needed.
dataDriveRoot = projectRoot / 'auxilliary'

In [ ]:
# --- Whole-trial ticks for change point figures ---
# Change points are trial numbers, so violin y-axes and heatmap colour bars
# only show whole trials. Call whole_trial_ticks() just before plt.show().
from matplotlib.collections import QuadMesh, PolyCollection
from matplotlib.ticker import MaxNLocator

def whole_trial_ticks(fig=None):
    fig = fig or plt.gcf()
    for ax in fig.axes:
        meshes = [c for c in ax.collections if isinstance(c, QuadMesh)]
        if meshes:                                   # heatmap -> integer colour bar
            cbar = meshes[0].colorbar
            if cbar is not None:
                cbar.locator = MaxNLocator(integer=True)
                cbar.update_ticks()
        elif any(isinstance(c, PolyCollection) for c in ax.collections):
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))   # violin

In [ ]:
#User defined params
regionType = 'SurfParcel'
#regionName = 'parcel'
preProcPipeline = 'Standard' # 'Standard', '025LPF, '015LPF', etc...

trialsIncluded = 'Full' # 'Hab', 'Nov, or 'Full'
firstTrial = 1
if trialsIncluded == 'Hab':
    lastTrial = 15
elif trialsIncluded == 'Nov':
    lastTrial = 20
else:
    lastTrial = 25

# Chromophore labels
CHROM_LABELS = {1: 'HbO', 2: 'HbR'}
CHROMS_TO_TEST = [1, 2]

exclude1mo = 1

activeOnly = 1

cptSignificance = 0.05 #alpha for determining the significance of a change point    
timeStart = 0
timeEnd = 19
numBasis = 21 #needs to match numBasis used when calculating changepoints
gridPoints = np.linspace(timeStart, timeEnd, num=191)

# --- Habituation-based filter ---
# 1 = restrict to infants who show significant masked habituation in at least
#     one of Fam1 vs Fam3, Fam1 vs Novel, or Fam1 vs Post, at that (Infant, Age,
#     Parcel, Chromophore).
# 0 = do not apply this filter.
habOnly = 0

# --- Change-point grouping for the paradigm-window plot ---
# Choose which change-point set to plot the distribution of. Options match
# the dictionaries populated further down (only "decreasing" categories are
# tracked - the "increasing" ones were dropped; never used downstream):
#   'first', 'final' (last), 'decreasing',
#   'first_and_decreasing', 'final_and_decreasing',
#   'earliest_decreasing', 'latest_decreasing'.
plotChangePointGroup = 'final_and_decreasing'


In [ ]:
inputFilenameChangepoint = projectRoot / 'data' / f'indigoIndividual{regionType}ChangepointDf{preProcPipeline}{trialsIncluded}.csv'
inputFilenameMetadata = projectRoot / 'dataRaw' / f'indigo{regionType}Metadata{preProcPipeline}.csv'

inputFilenameTimeseries = projectRoot / 'dataRaw' / f'indigo{regionType}TrialsVlen{preProcPipeline}.h5'

In [ ]:
# inputFilenameChangepoint

In [ ]:
changepointDf = pd.read_csv(inputFilenameChangepoint)
# changepointDf #look at first few rows
# with open(inputFilenameCoeffs, 'rb') as f:
#     regionChromCoeffs = pickle.load(f)

In [ ]:
# print(changepointDf)

In [ ]:
# change regionType name for downstream use if dealing with Surface Parcellation
if regionType == 'SurfParcel':
    regionType = 'Parcel'

    # Load the saved union habituation combinations
    inputFilenameHabCombos = projectRoot / 'data' / f'indigoIndividualSurfParcelHabCombosDf{preProcPipeline}{trialsIncluded}.csv'
    unionHabDf = pd.read_csv(inputFilenameHabCombos)
    unionHabDf['Parcel'] = unionHabDf['Parcel'].astype(int)
    unionHabDf['Infant'] = unionHabDf['Infant'].astype(changepointDf['Infant'].dtype)
    unionHabDf['Age'] = unionHabDf['Age'].astype(changepointDf['Age'].dtype)

    changepointDf = changepointDf.merge(
        unionHabDf[['Infant', 'Age', 'Parcel']].drop_duplicates(),
        on=['Infant', 'Age', 'Parcel'],
        how='inner'
    )

    if activeOnly == 1:
        # ---- Filter to TFCE-activation-significant combinations (HbO/HbR only) ----
        inputFilenameTFCESig = (
            projectRoot / 'data' /
            'surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv'
        )
        tfceSigDf = pd.read_csv(inputFilenameTFCESig)
        tfceSigDf = tfceSigDf[tfceSigDf['Significant'] == True][
            ['Age', 'Parcel', 'Chrom', 'Infant']
        ].drop_duplicates()
        tfceSigDf = tfceSigDf.rename(columns={'Chrom': 'Chromophore'})

        tfceSigDf['Parcel']      = tfceSigDf['Parcel'].astype(int)
        tfceSigDf['Infant']      = tfceSigDf['Infant'].astype(changepointDf['Infant'].dtype)
        tfceSigDf['Age']         = tfceSigDf['Age'].astype(changepointDf['Age'].dtype)
        tfceSigDf['Chromophore'] = tfceSigDf['Chromophore'].astype(changepointDf['Chromophore'].dtype)

        print(f"  changepointDf rows before TFCE activation filter: {len(changepointDf)}")
        changepointDf = changepointDf.merge(
            tfceSigDf,
            on=['Infant', 'Age', 'Parcel', 'Chromophore'],
            how='inner',
        )
        print(f"  changepointDf rows after TFCE activation filter:  {len(changepointDf)}")

        if changepointDf.empty:
            raise ValueError(
                "TFCE-activation-significance merge produced an empty dataframe."
            )

    if habOnly == 1:
        # ---- Filter to infants with masked-significant habituation in at least ----
        # ---- one of Fam1 vs Fam3, Fam1 vs Novel, or Fam1 vs Post ----
        habContrastFiles = [
            'surf_parcel_TFCE_habituation_MASKED_INDIV_spatiotemporal_perm_label.csv',
            'surf_parcel_TFCE_delayedHab_MASKED_INDIV_spatiotemporal_perm_label.csv',
            'surf_parcel_TFCE_lrHab_MASKED_INDIV_spatiotemporal_perm_label.csv',
        ]

        # Build the union of (Infant, Age, Parcel, Chromophore) tuples where
        # ANY habituation-type contrast reached masked significance.
        habUnionRows = []
        for fname in habContrastFiles:
            path = projectRoot / 'data' / fname
            if not path.exists():
                print(f"  WARNING: {fname} not found — skipping")
                continue
            tmpDf = pd.read_csv(path)
            if 'Masked significant' not in tmpDf.columns:
                print(f"  WARNING: 'Masked significant' column not in {fname} — skipping")
                continue
            tmpSig = tmpDf[tmpDf['Masked significant'] == True][
                ['Age', 'Parcel', 'Chrom', 'Infant']
            ].drop_duplicates()
            tmpSig = tmpSig.rename(columns={'Chrom': 'Chromophore'})
            habUnionRows.append(tmpSig)

        if habUnionRows:
            habUnionDf = pd.concat(habUnionRows, ignore_index=True).drop_duplicates()
            habUnionDf['Parcel']      = habUnionDf['Parcel'].astype(int)
            habUnionDf['Infant']      = habUnionDf['Infant'].astype(changepointDf['Infant'].dtype)
            habUnionDf['Age']         = habUnionDf['Age'].astype(changepointDf['Age'].dtype)
            habUnionDf['Chromophore'] = habUnionDf['Chromophore'].astype(changepointDf['Chromophore'].dtype)

            print(f"  changepointDf rows before habituation filter: {len(changepointDf)}")
            changepointDf = changepointDf.merge(
                habUnionDf,
                on=['Infant', 'Age', 'Parcel', 'Chromophore'],
                how='inner',
            )
            print(f"  changepointDf rows after habituation filter:  {len(changepointDf)}")

            if changepointDf.empty:
                raise ValueError(
                    "Habituation filter produced an empty dataframe — no infants "
                    "showed masked-significant habituation in any of the three contrasts."
                )
        else:
            raise FileNotFoundError(
                "No masked habituation results CSVs could be found. Check the file paths."
            )

In [ ]:
# ============================================================================
# Coverage check: of the Infant x {regionType} x Age x Chromophore combinations
# that COULD have been tested -- i.e. that pass the same eligibility filters
# used above (the mandatory habituation-combos filter via unionHabDf, plus
# activeOnly and/or habOnly) -- how many were actually excluded from the
# change-point pipeline before testing at all, due to missing/insufficient raw
# fNIRS data? This mirrors cell 5's filter sequence exactly (sequential
# intersection when both activeOnly and habOnly are enabled), but builds the
# eligible set independently of changepointDf, so combinations that never made
# it into changepointDf (because they were excluded pre-testing) are still
# counted here.
# ============================================================================

# ---- Load the exclusion log written by the change-point run ----
excludedDf = pd.read_csv(inputFileNameExcluded)
excludedDf['Infant'] = excludedDf['Infant'].astype(changepointDf['Infant'].dtype)
excludedDf[regionType] = excludedDf[regionType].astype(int)
excludedDf['Age'] = excludedDf['Age'].astype(changepointDf['Age'].dtype)
hasChromCol = 'Chrom' in excludedDf.columns  # only ever populated for Chrom Valid / Chrom Trials rows

# 'Combined Valid'/'Combined Trials' entries knock out the WHOLE Infant x
# {regionType} x Age combination (all chromophores), since that exclusion
# happens before the per-chromophore loop is ever reached.
combinedExcludedCombos = set(
    excludedDf[excludedDf['Log Type'].isin(['Combined Valid', 'Combined Trials'])]
    [['Infant', regionType, 'Age']].drop_duplicates()
    .itertuples(index=False, name=None)
)

# 'Chrom Valid'/'Chrom Trials' entries only knock out that specific chromophore
# for that combination (the combined HbT/HbDiff measures already succeeded).
if hasChromCol:
    chromExcludedRows = excludedDf[
        excludedDf['Log Type'].isin(['Chrom Valid', 'Chrom Trials']) & excludedDf['Chrom'].notna()
    ].copy()
    chromExcludedRows['Chrom'] = chromExcludedRows['Chrom'].astype(int)
    chromExcludedCombos = set(
        chromExcludedRows[['Infant', regionType, 'Age', 'Chrom']].drop_duplicates()
        .itertuples(index=False, name=None)
    )
else:
    print("WARNING: no 'Chrom' column found in the exclusion log -- treating "
          "'Chrom Valid'/'Chrom Trials' rows as knocking out the whole Infant x "
          f"{regionType} x Age combination, since the specific chromophore that "
          "failed can't be determined from this file.")
    chromExcludedCombos = set()
    combinedExcludedCombos |= set(
        excludedDf[excludedDf['Log Type'].isin(['Chrom Valid', 'Chrom Trials'])]
        [['Infant', regionType, 'Age']].drop_duplicates()
        .itertuples(index=False, name=None)
    )

print(f"Combined-level exclusions (whole combo lost): {len(combinedExcludedCombos)}")
print(f"Chromophore-level exclusions (single chromophore lost): {len(chromExcludedCombos)}")

# ---- Build the set of combinations that COULD have been tested, mirroring ----
# ---- cell 5's filter sequence exactly. ----
eligibleDf = unionHabDf[['Infant', 'Age', 'Parcel']].drop_duplicates().rename(columns={'Parcel': regionType})

if activeOnly == 1:
    inputFilenameTFCESig = (
        projectRoot / 'data' /
        'surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv'
    )
    tfceSigDf = pd.read_csv(inputFilenameTFCESig)
    tfceSigDf = tfceSigDf[tfceSigDf['Significant'] == True][
        ['Age', 'Parcel', 'Chrom', 'Infant']
    ].drop_duplicates()
    tfceSigDf = tfceSigDf.rename(columns={'Chrom': 'Chromophore', 'Parcel': regionType})
    eligibleDf = eligibleDf.merge(tfceSigDf, on=['Infant', 'Age', regionType], how='inner')

if habOnly == 1:
    habContrastFiles = [
        'surf_parcel_TFCE_habituation_MASKED_INDIV_spatiotemporal_perm_label.csv',
        'surf_parcel_TFCE_delayedHab_MASKED_INDIV_spatiotemporal_perm_label.csv',
        'surf_parcel_TFCE_lrHab_MASKED_INDIV_spatiotemporal_perm_label.csv',
    ]
    habUnionRows = []
    for fname in habContrastFiles:
        path = projectRoot / 'data' / fname
        if not path.exists():
            print(f"  WARNING: {fname} not found — skipping")
            continue
        tmpDf = pd.read_csv(path)
        if 'Masked significant' not in tmpDf.columns:
            print(f"  WARNING: 'Masked significant' column not in {fname} — skipping")
            continue
        tmpSig = tmpDf[tmpDf['Masked significant'] == True][
            ['Age', 'Parcel', 'Chrom', 'Infant']
        ].drop_duplicates()
        habUnionRows.append(tmpSig)
    if not habUnionRows:
        raise FileNotFoundError(
            "No masked habituation results CSVs could be found. Check the file paths."
        )
    habUnionDf = pd.concat(habUnionRows, ignore_index=True).drop_duplicates()
    habUnionDf = habUnionDf.rename(columns={'Chrom': 'Chromophore', 'Parcel': regionType})
    if 'Chromophore' in eligibleDf.columns:
        # activeOnly already ran and introduced Chromophore -- intersect on all 4 keys,
        # matching cell 5's sequential-AND behaviour when both toggles are on
        eligibleDf = eligibleDf.merge(habUnionDf, on=['Infant', 'Age', regionType, 'Chromophore'], how='inner')
    else:
        eligibleDf = eligibleDf.merge(habUnionDf, on=['Infant', 'Age', regionType], how='inner')

if 'Chromophore' not in eligibleDf.columns:
    raise ValueError(
        "Neither activeOnly nor habOnly is enabled -- Chromophore is never introduced, "
        "so there's no well-defined eligible set to check coverage against."
    )

eligibleDf = eligibleDf.drop_duplicates()
eligibleDf[regionType] = eligibleDf[regionType].astype(int)
eligibleDf['Infant'] = eligibleDf['Infant'].astype(changepointDf['Infant'].dtype)
eligibleDf['Age'] = eligibleDf['Age'].astype(changepointDf['Age'].dtype)
eligibleDf['Chromophore'] = eligibleDf['Chromophore'].astype(changepointDf['Chromophore'].dtype)

print(f"\nEligible (could-have-been-tested) combinations: {len(eligibleDf)} unique "
      f"Infant x {regionType} x Age x Chromophore rows.")

# ---- Flag each eligible combination as excluded (couldn't be tested) or not ----
def _wasExcluded(row):
    comboKey = (row['Infant'], row[regionType], row['Age'])
    chromKey = (row['Infant'], row[regionType], row['Age'], row['Chromophore'])
    return (comboKey in combinedExcludedCombos) or (chromKey in chromExcludedCombos)

eligibleDf['Excluded'] = eligibleDf.apply(_wasExcluded, axis=1)

# ---- Proportion that couldn't be tested, by age ----
proportionExcludedByAge = (
    eligibleDf.groupby('Age')['Excluded'].agg(['sum', 'count'])
    .rename(columns={'sum': 'N Excluded', 'count': 'N Eligible'})
)
proportionExcludedByAge['Proportion Excluded'] = (
    proportionExcludedByAge['N Excluded'] / proportionExcludedByAge['N Eligible']
)
print("\nProportion of eligible combinations that could NOT be tested, by age:")
print(proportionExcludedByAge)

# ---- List + save the specific combinations that couldn't be tested ----
outputFilenameCoverageGaps = projectRoot / 'data' / f'indigoIndividual{regionType}CoverageGapsDf{preProcPipeline}{trialsIncluded}.csv'
coverageGapsDf = (
    eligibleDf[eligibleDf['Excluded']]
    [['Infant', 'Age', regionType, 'Chromophore']]
    .sort_values(['Age', regionType, 'Infant', 'Chromophore'])
)
coverageGapsDf.to_csv(outputFilenameCoverageGaps, index=False)

print(f"\n{len(coverageGapsDf)} eligible combinations could not be tested; "
      f"saved to {outputFilenameCoverageGaps}")
coverageGapsDf


In [ ]:
# inputFilenameHabCombos

In [ ]:
changepointDf = changepointDf[changepointDf['Corrected p-value'] <= cptSignificance] #should be filtered already when generating, but just in case

uniqueAges = np.array(sorted(changepointDf['Age'].unique()))
if exclude1mo:
    uniqueAges = uniqueAges[1:]
uniqueChroms = changepointDf['Chromophore'].unique()
uniqueRegions = changepointDf[regionType].unique()

changepointDf['Decrease'] = 0
changepointDf['Change Point Count'] = 0
changepointDf['Change Size'] = 0
changepointDf['HRF Correlation'] = 0
changepointDf['HRF Covariance'] = 0

In [ ]:
metadataDf = pd.read_csv(inputFilenameMetadata)

In [ ]:
# Initialise dictionaries
# NOTE: only the "decreasing" change-point categories are tracked here - these
# are the only ones reported in the paper (Table 2: Decreasing, First AND
# decreasing, Final AND decreasing, Earliest decreasing, Latest decreasing).
# The corresponding "increasing" categories were dropped entirely (never used
# downstream in either this notebook or its non-chromophore-specific sibling).
firstChangePoints_dict = {}
lastChangePoints_dict = {}
decrChangePoints_dict = {}
firstAndDecrChangePoints_dict = {}
finalAndDecrChangePoints_dict = {}
latestDecrChangePoints_dict = {}
earliestDecrChangePoints_dict = {}
countPartRegion_dict = {}
ageCPtDFs = {}


In [ ]:

basis = BSplineBasis(n_basis=numBasis)


In [ ]:

# load data
with h5py.File(inputFilenameTimeseries, 'r') as h5f:
    tsDataset = h5f['timeSeries'][()].copy()


In [ ]:
if activeOnly and regionType == 'Tile':

    changepointDf = changepointDf[changepointDf[regionType].isin([2,8])]

# Set up evaluation grid and canonical HRF to compare curves to
timeStart, timeEnd = 0, 19
basis = BSplineBasis(domain_range=(timeStart, timeEnd), n_basis=numBasis)

for age in uniqueAges:

    ageDf = changepointDf[changepointDf['Age'] == age]

    for chrom in uniqueChroms:

        chromDf = ageDf[ageDf['Chromophore'] == chrom]

        for region in uniqueRegions:

            regionDf = chromDf[chromDf[regionType] == region]

            if regionDf.empty: # no data to extract
                continue

            uniqueInfants = regionDf['Infant'].drop_duplicates().values

            for infant in uniqueInfants:

                # print(age, chrom, region, infant)

                infantDf = regionDf[regionDf['Infant'] == infant].sort_values(by=['Change Point'])

                changeColumn = [int(x) for x in infantDf.get('Change Point').tolist()]
                changeColumn.append(lastTrial + 1)      

                comboDf = metadataDf[
                    (metadataDf['Age'] == age) &
                    (metadataDf['Chrom'] == chrom) &
                    (metadataDf[regionType] == region) & 
                    (metadataDf['Infant'] == infant)
                ]

                trialList = np.array(sorted(comboDf['Trial'].values))
                tsList = [tsDataset[tsIDx] for tsIDx in comboDf['tsIDx']]

                # check for, and remove, NaN rows
                filteredTrialInfo = [
                    (ts, trial) for ts, trial in zip(tsList, trialList)
                    if not np.isnan(ts).any()
                ]
                if len(filteredTrialInfo) == 0:
                    # print(uniqueInfants)
                    # print(infant)
                    # print(comboDf)
                    # print('ZERO')
                    continue  # skip this infant — no valid time series
                # print('NOT ZERO')
                tsList, trialList = zip(*filteredTrialInfo)
                minLength = min(ts.shape[-1] for ts in tsList)
                tsList = np.stack([ts[..., :minLength] for ts in tsList])
                trialList = np.array(trialList)
                
                
                # move on if time series data non-existent or populated wholly by NaNs
                # if np.shape(tsList)[0] == 0: #shouldn't be the case, but just in case
                #     continue

                # Initialise variables for mean curve calculations betw. changeColumn
                iBlock = 0
                firstBlockTrial = 1
                blockMeans = np.empty([len(changeColumn), np.shape(tsList)[1]])

                # Calculate mean trials between CPts
                for point in changeColumn:
                    
                    lastBlockTrial = point-1 #-1 because dataMatrix rows go from 0->(numTrials-1)
                    includedRows = (trialList >= firstBlockTrial) & (trialList < lastBlockTrial)
                    if not includedRows.any():
                        continue  # skip this block
                    meanBlock = np.nanmean(tsList[includedRows], axis=0)
                    if np.isnan(meanBlock).any():
                        continue  # skip this block if it failed
                    blockMeans[iBlock] = meanBlock
                    firstBlockTrial = point
                    iBlock = iBlock + 1

                #
                evalGrid = np.linspace(timeStart, timeEnd, np.shape(tsList)[1])
                canonical = fNIRSData.canonicalHrf(evalGrid, double = 0, onset=0, duration=evalGrid[-1], peakDelay = 9)
                canonical = 0.5e-6 * (canonical / np.max(np.abs(canonical)))

                # Calculate absolute block means
                absBlockMeans = np.abs(blockMeans)

                for i in range(1, len(changeColumn)):

                    #for inserting change size/type values into right place:
                    c1 = changepointDf[regionType] == region #tile, parcel, channel number...
                    c2 = changepointDf['Change Point'] == changeColumn[i-1]
                    c3 = changepointDf['Age'] == age
                    c4 = changepointDf['Chromophore'] == chrom
                    c5 = changepointDf['Infant'] == infant
                    mask = c1 & c2 & c3 & c4 & c5

                    # Calculate 'piece' of volume between two current CPts
                    aucMeanTrialBefore = simpson(absBlockMeans[i-1,:], dx = 1)
                    aucMeanTrialAfter = simpson(absBlockMeans[i,:], dx = 1)

                    # Compute change size
                    changeSize = aucMeanTrialBefore - aucMeanTrialAfter
                    changepointDf.loc[mask, 'Change Size'] = changeSize

                    # Change the indicator in the 'Decrease' column if the magnitude of the response Decreases after the change point
                    if aucMeanTrialAfter < aucMeanTrialBefore:
                        changepointDf.loc[mask, 'Decrease'] = 1
                    else:
                        changepointDf.loc[mask, 'Decrease'] = 0

                    # compute correlation with canonical HRF
                    yValsBefore = blockMeans[i-1,:]
                    corr, _ = pearsonr(yValsBefore, canonical)
                    changepointDf.loc[mask, 'HRF Correlation'] = corr

                    # compute covariance with canonical HRF
                    covVal = np.cov(yValsBefore*1e7, canonical*1e7)[0, 1]
                    changepointDf.loc[mask, 'HRF Covariance'] = covVal

                    # #add change point count to relevant location
                    # changepointDf.loc[mask, "Change Point Count"] = cPtCount[region]

                # plt.figure()

                # for i in range(blockMeans.shape[0]):
                #     yVals = blockMeans[i,:]
                #     if np.isnan(yVals).any():
                #         continue  # skip this block if it failed

                #     # Compute correlation with canonical HRF
                #     corr, _ = pearsonr(yVals, canonical)
                #     covVal = np.cov(yVals*1e7, canonical*1e7)[0, 1]
                #     corrStr = f"{corr:.2f}" if not np.isnan(corr) else "N/A"
                #     covStr = f"{covVal:.2e}" if not np.isnan(covVal) else "N/A"
                #     labelStr = f'Block {i+1} (Corr: {corrStr})'
                #     labelStr = f'Block {i+1} (Corr: {corrStr}, Cov: {covStr})'

                #     plt.plot(evalGrid, yVals, label=labelStr)

                # plt.title(f'Infant {infant} | Age {age} | Region {region} | Chrom {chrom}')
                # plt.xlabel('Time (s)')
                # plt.ylabel('Mean Concentration Change (M)')
                # plt.xticks(np.arange(20)) 
                # plt.legend(title='Block / Canonical Corr')
                # plt.tight_layout()
                # plt.show()
                

In [ ]:
if regionType == 'Channel':
    groupCols = ['Age', 'Source', 'Detector', 'Chromophore']
else:
    groupCols = ['Age', regionType, 'Chromophore']

# find which combinations have any row meeting both criteria (correlation and covariance with 'canonical' response)
validGroups = (
    changepointDf
    .groupby(groupCols)
    .filter(lambda df: (df['HRF Correlation'].abs() > 0.6).any() and (df['HRF Covariance'].abs() > 1.0).any())
)

# Replace original dataframe with filtered version
changepointDf = validGroups.copy()
print(changepointDf.head())

In [ ]:
for age in uniqueAges:

    dfCPts = changepointDf[changepointDf['Age'] == age]

    # Find IDs of first change points for each region in the filtered DF
    idx = dfCPts.groupby([regionType, 'Chromophore', 'Infant'])['Change Point'].idxmin()
    #Slice CPts DF with just these IDs
    dfFirstCPts = dfCPts.loc[idx]
    firstChangePoints_dict[age] = dfFirstCPts

    # Find IDs of last change points for each region in the filtered DF
    idx = dfCPts.groupby([regionType, 'Chromophore', 'Infant'])['Change Point'].idxmax()
    #Slice CPts DF with just these IDs
    dfLastCPts = dfCPts.loc[idx]
    lastChangePoints_dict[age] = dfLastCPts

    # Slice CPts DF containing only 'decreasing' changepoints
    dfDecreasingCPts = dfCPts[dfCPts['Decrease'] == 1]
    decrChangePoints_dict[age] = dfDecreasingCPts

    #Slice CPts DF containing only first AND decreasing CPts
    dfIntersection = dfCPts[dfCPts.index.isin(dfFirstCPts.index) & dfCPts.index.isin(dfDecreasingCPts.index)]
    firstAndDecrChangePoints_dict[age] = dfIntersection

    #Slice CPts DF containing only last AND decreasing CPts
    dfIntersection = dfCPts[dfCPts.index.isin(dfLastCPts.index) & dfCPts.index.isin(dfDecreasingCPts.index)]
    finalAndDecrChangePoints_dict[age] = dfIntersection

    # Find IDs of last change points for each region in the DF containing 
    # ONLY decreasing changepoints
    idx = dfDecreasingCPts.groupby([regionType, 'Chromophore', 'Infant'])['Change Point'].idxmax()
    #Slice CPts DF with just these IDs
    dfLatestDecrCPts = dfDecreasingCPts.loc[idx]
    latestDecrChangePoints_dict[age] = dfLatestDecrCPts

    # find IDs of first change points for each region in the DF containing 
    # ONLY decreasing changepoints
    idx = dfDecreasingCPts.groupby([regionType, 'Chromophore', 'Infant'])['Change Point'].idxmin()
    #Slice CPts DF with just these IDs
    dfEarliestDecrCPts = dfDecreasingCPts.loc[idx]
    earliestDecrChangePoints_dict[age] = dfEarliestDecrCPts

    # view CPts dataframe in nicer format 
    dfAgeName = f"CPt Dataframe: {age}mo"
    ageCPtDFs[dfAgeName] = dfCPts

# Create DFs combining data from all ages
all_firstChangePoints = pd.concat(firstChangePoints_dict, ignore_index=True)
all_lastChangePoints = pd.concat(lastChangePoints_dict, ignore_index=True)
all_decrChangePoints = pd.concat(decrChangePoints_dict, ignore_index=True)
all_firstAndDecrChangePoints = pd.concat(firstAndDecrChangePoints_dict, ignore_index=True)
all_finalAndDecrChangePoints = pd.concat(finalAndDecrChangePoints_dict, ignore_index=True)
all_latestDecrChangePoints = pd.concat(latestDecrChangePoints_dict, ignore_index=True)
all_earliestDecrChangePoints = pd.concat(earliestDecrChangePoints_dict, ignore_index=True)


In [ ]:
# Concatenate all age data
dfAllAges = pd.concat(ageCPtDFs.values(), ignore_index=True)
dfAllAges = dfAllAges[dfAllAges["Change Size"].abs() < 0.0001]

# Define unique age values and a colormap
colors = plt.cm.viridis(np.linspace(0, 1, len(uniqueAges)))  # Get distinct colors

# Create figure
plt.figure(figsize=(10, 6))

# Plot each age group separately with a specific color
for age, color in zip(uniqueAges, colors):
    subset = dfAllAges[dfAllAges["Age"] == age]
    plt.scatter(subset["Change Point"], subset["Change Size"]*10000, color=color, label=f"Age {age}", alpha=0.5)

plt.xlabel("Change Point Number")
plt.ylabel("Change Size ($\mu$M$^2$)")
plt.title("Change Point Number vs. Change Size")

ax = plt.gca()  # Get current axis
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)

# Add a legend instead of a colorbar
plt.legend(title="Age", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.show()


In [ ]:
# Concatenate all age data
# dfAllAges = pd.concat(ageCPtDFs.values(), ignore_index=True)

# Convert change size to μM² units (if not already scaled)
dfAllAges["Change Size (μM²)"] = dfAllAges["Change Size"] * 10000

plt.figure(figsize=(10, 6))
sns.histplot(
    data=dfAllAges,
    x="Change Size (μM²)",
    hue="Age",
    multiple="dodge",  # use 'layer' to stack instead
    palette="viridis",
    bins=30,
    stat="density",  # So height reflects density, like KDE
    edgecolor="black",
    alpha=0.6
)

plt.xlabel("Change Size ($\mu$M$^2$)")
plt.ylabel("Density")
plt.title("Change Size Distributions by Age (Histogram)")
sns.despine()
plt.tight_layout()
plt.show()



In [ ]:
# # Print dictionaries if needed
# print(lastDetect_dict)
# print(lastChanChanges_dict)
# print(finalAndDecrDetect_dict)
# print(finalAndDecrChanChanges_dict)
# print(latestDecrDetect_dict)
# print(latestDecrChanChanges_dict)

In [ ]:
# Storage for statistical test results, keyed by test name.
# Populated by each test cell as it runs; read by the summary cell.
statTestResults = {}

In [ ]:
# Decreasing CPt data - per chromophore
print("STATISTICAL ANALYSIS FOR: Decreasing CPt data")
print(" ")

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]
    print(f"\n{'=' * 60}")
    print(f"  Chromophore: {chrom_label}")
    print(f"{'=' * 60}")

    sub_df = all_decrChangePoints[
        all_decrChangePoints['Chromophore'] == chrom
    ]

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    n_obs = len(df)
    n_unique_ages = df['Timepoint'].nunique() if n_obs > 0 else 0
    n_unique_outcomes = df['ChangepointTrial'].nunique() if n_obs > 0 else 0

    if n_obs < 5 or n_unique_ages < 2 or n_unique_outcomes < 2:
        print(f"  SKIPPING: n_obs={n_obs}, n_unique_ages={n_unique_ages}, "
              f"n_unique_outcomes={n_unique_outcomes}")
        statTestResults[('Decreasing', chrom_label)] = {
            'n_obs': n_obs, 'n_ages': n_unique_ages,
            'ologit_coef': float('nan'), 'ologit_pvalue': float('nan'),
            'wald_statistic': float('nan'), 'wald_pvalue': float('nan'),
            'tukey_meandiff': float('nan'), 'tukey_pvalue': float('nan'),
            'tukey_reject': False,
        }
        continue

    model_ologit = OrderedModel(df['ChangepointTrial'],
                                 df[['Timepoint']],
                                 distr='logit')
    result_ologit = model_ologit.fit(method='newton')
    print("Ordinal Logistic Regression:")
    print(result_ologit.summary())
    print()

    wald_test_timepoint = result_ologit.wald_test('Timepoint')
    print("Outcome of conducting a post-hoc Wald test:")
    print(wald_test_timepoint)
    print()

    tukey_results = pairwise_tukeyhsd(endog=df['ChangepointTrial'],
                                       groups=df['Timepoint'],
                                       alpha=0.05)
    print(tukey_results)

    statTestResults[('Decreasing', chrom_label)] = {
        'n_obs': len(df),
        'n_ages': df['Timepoint'].nunique(),
        'ologit_coef': result_ologit.params.get('Timepoint', float('nan')),
        'ologit_pvalue': result_ologit.pvalues.get('Timepoint', float('nan')),
        'wald_statistic': float(wald_test_timepoint.statistic.item()) if hasattr(wald_test_timepoint.statistic, 'item') else float(wald_test_timepoint.statistic),
        'wald_pvalue': float(wald_test_timepoint.pvalue),
        'tukey_meandiff': float(tukey_results.meandiffs[0]) if len(tukey_results.meandiffs) > 0 else float('nan'),
        'tukey_pvalue': float(tukey_results.pvalues[0]) if len(tukey_results.pvalues) > 0 else float('nan'),
        'tukey_reject': bool(tukey_results.reject[0]) if len(tukey_results.reject) > 0 else False,
    }

In [ ]:
# --- Decreasing: 4 figures (HbO violin, HbR violin, HbO heatmap, HbR heatmap) ---

palette_violin = sns.color_palette("bright", 3)
paletteHeatmap = 'cividis_r'

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]

    sub_df = all_decrChangePoints[
        all_decrChangePoints['Chromophore'] == chrom
    ].copy()

    if sub_df.empty:
        print(f"  {chrom_label}: no data — skipping")
        continue

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    # Violin plot
    df_filtered = df.dropna(subset=['ChangepointTrial']).loc[df['ChangepointTrial'] != 0]
    plt.figure(figsize=(10, 8))
    sns.violinplot(x='Timepoint', y='ChangepointTrial', data=df_filtered,
                    palette=palette_violin)
    plt.title(f'Decreasing Changepoints Distribution by Age (months) — {chrom_label}')
    plt.ylabel('Changepoint')
    whole_trial_ticks()
    plt.show()

    # Heatmap
    pivot_table = df.pivot_table(
        index=regionType, columns='Timepoint',
        values='ChangepointTrial', aggfunc=np.mean,
    )
    pivot_table_no_zeros = pivot_table.replace(0, np.nan)
    pivot_table_no_zeros = pivot_table_no_zeros.sort_index()

    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table_no_zeros, cmap=paletteHeatmap,
                 annot=True, fmt='.0f', cbar=True)
    plt.title(f'Average Decreasing Changepoints Heatmap by {regionType} and Timepoint — {chrom_label}')
    plt.xlabel('Age')
    plt.ylabel(regionType)
    whole_trial_ticks()
    plt.show()

In [ ]:
# First AND Decreasing CPt data - per chromophore
print("STATISTICAL ANALYSIS FOR: First AND decreasing CPt data")
print(" ")

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]
    print(f"\n{'=' * 60}")
    print(f"  Chromophore: {chrom_label}")
    print(f"{'=' * 60}")

    sub_df = all_firstAndDecrChangePoints[
        all_firstAndDecrChangePoints['Chromophore'] == chrom
    ]

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    n_obs = len(df)
    n_unique_ages = df['Timepoint'].nunique() if n_obs > 0 else 0
    n_unique_outcomes = df['ChangepointTrial'].nunique() if n_obs > 0 else 0

    if n_obs < 5 or n_unique_ages < 2 or n_unique_outcomes < 2:
        print(f"  SKIPPING: n_obs={n_obs}, n_unique_ages={n_unique_ages}, "
              f"n_unique_outcomes={n_unique_outcomes}")
        statTestResults[('First AND decreasing', chrom_label)] = {
            'n_obs': n_obs, 'n_ages': n_unique_ages,
            'ologit_coef': float('nan'), 'ologit_pvalue': float('nan'),
            'wald_statistic': float('nan'), 'wald_pvalue': float('nan'),
            'tukey_meandiff': float('nan'), 'tukey_pvalue': float('nan'),
            'tukey_reject': False,
        }
        continue

    model_ologit = OrderedModel(df['ChangepointTrial'],
                                 df[['Timepoint']],
                                 distr='logit')
    result_ologit = model_ologit.fit(method='newton')
    print("Ordinal Logistic Regression:")
    print(result_ologit.summary())
    print()

    wald_test_timepoint = result_ologit.wald_test('Timepoint')
    print("Outcome of conducting a post-hoc Wald test:")
    print(wald_test_timepoint)
    print()

    tukey_results = pairwise_tukeyhsd(endog=df['ChangepointTrial'],
                                       groups=df['Timepoint'],
                                       alpha=0.05)
    print(tukey_results)

    statTestResults[('First AND decreasing', chrom_label)] = {
        'n_obs': len(df),
        'n_ages': df['Timepoint'].nunique(),
        'ologit_coef': result_ologit.params.get('Timepoint', float('nan')),
        'ologit_pvalue': result_ologit.pvalues.get('Timepoint', float('nan')),
        'wald_statistic': float(wald_test_timepoint.statistic.item()) if hasattr(wald_test_timepoint.statistic, 'item') else float(wald_test_timepoint.statistic),
        'wald_pvalue': float(wald_test_timepoint.pvalue),
        'tukey_meandiff': float(tukey_results.meandiffs[0]) if len(tukey_results.meandiffs) > 0 else float('nan'),
        'tukey_pvalue': float(tukey_results.pvalues[0]) if len(tukey_results.pvalues) > 0 else float('nan'),
        'tukey_reject': bool(tukey_results.reject[0]) if len(tukey_results.reject) > 0 else False,
    }

In [ ]:
# --- First AND decreasing: 4 figures (HbO violin, HbR violin, HbO heatmap, HbR heatmap) ---

palette_violin = sns.color_palette("bright", 3)
paletteHeatmap = 'cividis_r'

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]

    sub_df = all_firstAndDecrChangePoints[
        all_firstAndDecrChangePoints['Chromophore'] == chrom
    ].copy()

    if sub_df.empty:
        print(f"  {chrom_label}: no data — skipping")
        continue

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    df_filtered = df.dropna(subset=['ChangepointTrial']).loc[df['ChangepointTrial'] != 0]
    plt.figure(figsize=(10, 8))
    sns.violinplot(x='Timepoint', y='ChangepointTrial', data=df_filtered,
                    palette=palette_violin)
    plt.title(f'First AND decreasing Changepoints Distribution by Age (months) — {chrom_label}')
    plt.ylabel('Changepoint')
    whole_trial_ticks()
    plt.show()

    pivot_table = df.pivot_table(
        index=regionType, columns='Timepoint',
        values='ChangepointTrial', aggfunc=np.mean,
    )
    pivot_table_no_zeros = pivot_table.replace(0, np.nan)
    pivot_table_no_zeros = pivot_table_no_zeros.sort_index()

    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table_no_zeros, cmap=paletteHeatmap,
                 annot=True, fmt='.0f', cbar=True)
    plt.title(f'Average First AND decreasing Changepoints Heatmap by {regionType} and Timepoint — {chrom_label}')
    plt.xlabel('Age')
    plt.ylabel(regionType)
    whole_trial_ticks()
    plt.show()

In [ ]:
# Final AND Decreasing CPt data - per chromophore
print("STATISTICAL ANALYSIS FOR: Final AND decreasing CPt data")
print(" ")

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]
    print(f"\n{'=' * 60}")
    print(f"  Chromophore: {chrom_label}")
    print(f"{'=' * 60}")

    sub_df = all_finalAndDecrChangePoints[
        all_finalAndDecrChangePoints['Chromophore'] == chrom
    ]

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    # Skip if the model can't be fitted meaningfully
    n_obs = len(df)
    n_unique_ages = df['Timepoint'].nunique() if n_obs > 0 else 0
    n_unique_outcomes = df['ChangepointTrial'].nunique() if n_obs > 0 else 0

    if n_obs < 5 or n_unique_ages < 2 or n_unique_outcomes < 2:
        print(f"  SKIPPING: n_obs={n_obs}, n_unique_ages={n_unique_ages}, "
              f"n_unique_outcomes={n_unique_outcomes}")
        # Store a NaN placeholder so the summary cell can note the skip
        statTestResults[('Final AND decreasing', chrom_label)] = {
            'n_obs': n_obs, 'n_ages': n_unique_ages,
            'ologit_coef': float('nan'), 'ologit_pvalue': float('nan'),
            'wald_statistic': float('nan'), 'wald_pvalue': float('nan'),
            'tukey_meandiff': float('nan'), 'tukey_pvalue': float('nan'),
            'tukey_reject': False,
        }
        continue

    # Ordinal Logistic Regression
    model_ologit = OrderedModel(df['ChangepointTrial'],
                                 df[['Timepoint']],
                                 distr='logit')
    result_ologit = model_ologit.fit(method='newton')
    print("Ordinal Logistic Regression:")
    print(result_ologit.summary())
    print()

    # Wald test
    wald_test_timepoint = result_ologit.wald_test('Timepoint')
    print("Outcome of conducting a post-hoc Wald test:")
    print(wald_test_timepoint)
    print()

    # Tukey HSD
    tukey_results = pairwise_tukeyhsd(endog=df['ChangepointTrial'],
                                       groups=df['Timepoint'],
                                       alpha=0.05)
    print(tukey_results)

    # Store results keyed by (test_name, chromophore)
    statTestResults[('Final AND decreasing', chrom_label)] = {
        'n_obs': len(df),
        'n_ages': df['Timepoint'].nunique(),
        'ologit_coef': result_ologit.params.get('Timepoint', float('nan')),
        'ologit_pvalue': result_ologit.pvalues.get('Timepoint', float('nan')),
        'wald_statistic': float(wald_test_timepoint.statistic.item()) if hasattr(wald_test_timepoint.statistic, 'item') else float(wald_test_timepoint.statistic),
        'wald_pvalue': float(wald_test_timepoint.pvalue),
        'tukey_meandiff': float(tukey_results.meandiffs[0]) if len(tukey_results.meandiffs) > 0 else float('nan'),
        'tukey_pvalue': float(tukey_results.pvalues[0]) if len(tukey_results.pvalues) > 0 else float('nan'),
        'tukey_reject': bool(tukey_results.reject[0]) if len(tukey_results.reject) > 0 else False,
    }

In [ ]:
# --- Final AND decreasing: 4 figures (HbO violin, HbR violin, HbO heatmap, HbR heatmap) ---

palette_violin = sns.color_palette("bright", 3)
paletteHeatmap = 'cividis_r'

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]

    sub_df = all_finalAndDecrChangePoints[
        all_finalAndDecrChangePoints['Chromophore'] == chrom
    ].copy()

    if sub_df.empty:
        print(f"  {chrom_label}: no data — skipping")
        continue

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    df_filtered = df.dropna(subset=['ChangepointTrial']).loc[df['ChangepointTrial'] != 0]
    plt.figure(figsize=(10, 8))
    sns.violinplot(x='Timepoint', y='ChangepointTrial', data=df_filtered,
                    palette=palette_violin)
    plt.title(f'Final AND decreasing Changepoints Distribution by Age (months) — {chrom_label}')
    plt.ylabel('Changepoint')
    whole_trial_ticks()
    plt.show()

    pivot_table = df.pivot_table(
        index=regionType, columns='Timepoint',
        values='ChangepointTrial', aggfunc=np.mean,
    )
    pivot_table_no_zeros = pivot_table.replace(0, np.nan)
    pivot_table_no_zeros = pivot_table_no_zeros.sort_index()

    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table_no_zeros, cmap=paletteHeatmap,
                 annot=True, fmt='.0f', cbar=True)
    plt.title(f'Average Final AND decreasing Changepoints Heatmap by {regionType} and Timepoint — {chrom_label}')
    plt.xlabel('Age')
    plt.ylabel(regionType)
    whole_trial_ticks()
    plt.show()

In [ ]:
df_filtered = df.dropna(subset=['ChangepointTrial']).loc[df['ChangepointTrial'] != 0]

# Colorblind-friendly palette
palette_violin = ["#0072B2", "#D55E00", "#009E73"]  # blue, vermillion, green
# palette_violin = sns.color_palette("Set2", 3)
# palette_violin = sns.color_palette("deep", 3)


plt.figure(figsize=(10, 8))

# Violin (background)
sns.violinplot(
    x='Timepoint', y='ChangepointTrial', data=df_filtered,
    palette=palette_violin, inner=None, linewidth=1,
)

# Strip (individual points)
sns.stripplot(
    x='Timepoint', y='ChangepointTrial', data=df_filtered,
    color='black', size=6, alpha=0.6, jitter=0.1,
)

plt.title('Final AND decreasing Changepoints Distribution by Age (months)')
plt.ylabel('Changepoint')
whole_trial_ticks()
plt.show()

In [ ]:
# Earliest Decreasing CPt data - per chromophore
print("STATISTICAL ANALYSIS FOR: Earliest decreasing CPt data")
print(" ")

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]
    print(f"\n{'=' * 60}")
    print(f"  Chromophore: {chrom_label}")
    print(f"{'=' * 60}")

    sub_df = all_earliestDecrChangePoints[
        all_earliestDecrChangePoints['Chromophore'] == chrom
    ]

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    n_obs = len(df)
    n_unique_ages = df['Timepoint'].nunique() if n_obs > 0 else 0
    n_unique_outcomes = df['ChangepointTrial'].nunique() if n_obs > 0 else 0

    if n_obs < 5 or n_unique_ages < 2 or n_unique_outcomes < 2:
        print(f"  SKIPPING: n_obs={n_obs}, n_unique_ages={n_unique_ages}, "
              f"n_unique_outcomes={n_unique_outcomes}")
        statTestResults[('Earliest decreasing', chrom_label)] = {
            'n_obs': n_obs, 'n_ages': n_unique_ages,
            'ologit_coef': float('nan'), 'ologit_pvalue': float('nan'),
            'wald_statistic': float('nan'), 'wald_pvalue': float('nan'),
            'tukey_meandiff': float('nan'), 'tukey_pvalue': float('nan'),
            'tukey_reject': False,
        }
        continue

    model_ologit = OrderedModel(df['ChangepointTrial'],
                                 df[['Timepoint']],
                                 distr='logit')
    result_ologit = model_ologit.fit(method='newton')
    print("Ordinal Logistic Regression:")
    print(result_ologit.summary())
    print()

    wald_test_timepoint = result_ologit.wald_test('Timepoint')
    print("Outcome of conducting a post-hoc Wald test:")
    print(wald_test_timepoint)
    print()

    tukey_results = pairwise_tukeyhsd(endog=df['ChangepointTrial'],
                                       groups=df['Timepoint'],
                                       alpha=0.05)
    print(tukey_results)

    statTestResults[('Earliest decreasing', chrom_label)] = {
        'n_obs': len(df),
        'n_ages': df['Timepoint'].nunique(),
        'ologit_coef': result_ologit.params.get('Timepoint', float('nan')),
        'ologit_pvalue': result_ologit.pvalues.get('Timepoint', float('nan')),
        'wald_statistic': float(wald_test_timepoint.statistic.item()) if hasattr(wald_test_timepoint.statistic, 'item') else float(wald_test_timepoint.statistic),
        'wald_pvalue': float(wald_test_timepoint.pvalue),
        'tukey_meandiff': float(tukey_results.meandiffs[0]) if len(tukey_results.meandiffs) > 0 else float('nan'),
        'tukey_pvalue': float(tukey_results.pvalues[0]) if len(tukey_results.pvalues) > 0 else float('nan'),
        'tukey_reject': bool(tukey_results.reject[0]) if len(tukey_results.reject) > 0 else False,
    }

In [ ]:
# --- Earliest decreasing: 4 figures (HbO violin, HbR violin, HbO heatmap, HbR heatmap) ---

palette_violin = sns.color_palette("bright", 3)
paletteHeatmap = 'cividis_r'

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]

    sub_df = all_earliestDecrChangePoints[
        all_earliestDecrChangePoints['Chromophore'] == chrom
    ].copy()

    if sub_df.empty:
        print(f"  {chrom_label}: no data — skipping")
        continue

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    df_filtered = df.dropna(subset=['ChangepointTrial']).loc[df['ChangepointTrial'] != 0]
    plt.figure(figsize=(10, 8))
    sns.violinplot(x='Timepoint', y='ChangepointTrial', data=df_filtered,
                    palette=palette_violin)
    plt.title(f'Earliest decreasing Changepoints Distribution by Age (months) — {chrom_label}')
    plt.ylabel('Changepoint')
    whole_trial_ticks()
    plt.show()

    pivot_table = df.pivot_table(
        index=regionType, columns='Timepoint',
        values='ChangepointTrial', aggfunc=np.mean,
    )
    pivot_table_no_zeros = pivot_table.replace(0, np.nan)
    pivot_table_no_zeros = pivot_table_no_zeros.sort_index()

    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table_no_zeros, cmap=paletteHeatmap,
                 annot=True, fmt='.0f', cbar=True)
    plt.title(f'Average Earliest decreasing Changepoints Heatmap by {regionType} and Timepoint — {chrom_label}')
    plt.xlabel('Age')
    plt.ylabel(regionType)
    whole_trial_ticks()
    plt.show()

In [ ]:
# Latest Decreasing CPt data - per chromophore
print("STATISTICAL ANALYSIS FOR: Latest decreasing CPt data")
print(" ")

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]
    print(f"\n{'=' * 60}")
    print(f"  Chromophore: {chrom_label}")
    print(f"{'=' * 60}")

    sub_df = all_latestDecrChangePoints[
        all_latestDecrChangePoints['Chromophore'] == chrom
    ]

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    n_obs = len(df)
    n_unique_ages = df['Timepoint'].nunique() if n_obs > 0 else 0
    n_unique_outcomes = df['ChangepointTrial'].nunique() if n_obs > 0 else 0

    if n_obs < 5 or n_unique_ages < 2 or n_unique_outcomes < 2:
        print(f"  SKIPPING: n_obs={n_obs}, n_unique_ages={n_unique_ages}, "
              f"n_unique_outcomes={n_unique_outcomes}")
        statTestResults[('Latest decreasing', chrom_label)] = {
            'n_obs': n_obs, 'n_ages': n_unique_ages,
            'ologit_coef': float('nan'), 'ologit_pvalue': float('nan'),
            'wald_statistic': float('nan'), 'wald_pvalue': float('nan'),
            'tukey_meandiff': float('nan'), 'tukey_pvalue': float('nan'),
            'tukey_reject': False,
        }
        continue

    model_ologit = OrderedModel(df['ChangepointTrial'],
                                 df[['Timepoint']],
                                 distr='logit')
    result_ologit = model_ologit.fit(method='newton')
    print("Ordinal Logistic Regression:")
    print(result_ologit.summary())
    print()

    wald_test_timepoint = result_ologit.wald_test('Timepoint')
    print("Outcome of conducting a post-hoc Wald test:")
    print(wald_test_timepoint)
    print()

    tukey_results = pairwise_tukeyhsd(endog=df['ChangepointTrial'],
                                       groups=df['Timepoint'],
                                       alpha=0.05)
    print(tukey_results)

    statTestResults[('Latest decreasing', chrom_label)] = {
        'n_obs': len(df),
        'n_ages': df['Timepoint'].nunique(),
        'ologit_coef': result_ologit.params.get('Timepoint', float('nan')),
        'ologit_pvalue': result_ologit.pvalues.get('Timepoint', float('nan')),
        'wald_statistic': float(wald_test_timepoint.statistic.item()) if hasattr(wald_test_timepoint.statistic, 'item') else float(wald_test_timepoint.statistic),
        'wald_pvalue': float(wald_test_timepoint.pvalue),
        'tukey_meandiff': float(tukey_results.meandiffs[0]) if len(tukey_results.meandiffs) > 0 else float('nan'),
        'tukey_pvalue': float(tukey_results.pvalues[0]) if len(tukey_results.pvalues) > 0 else float('nan'),
        'tukey_reject': bool(tukey_results.reject[0]) if len(tukey_results.reject) > 0 else False,
    }

In [ ]:
# --- Latest decreasing: 4 figures (HbO violin, HbR violin, HbO heatmap, HbR heatmap) ---

palette_violin = sns.color_palette("bright", 3)
paletteHeatmap = 'cividis_r'

for chrom in CHROMS_TO_TEST:
    chrom_label = CHROM_LABELS[chrom]

    sub_df = all_latestDecrChangePoints[
        all_latestDecrChangePoints['Chromophore'] == chrom
    ].copy()

    if sub_df.empty:
        print(f"  {chrom_label}: no data — skipping")
        continue

    data = {
        regionType: sub_df[regionType].values.astype(int),
        'Timepoint': sub_df['Age'].values.astype(int),
        'ChangepointTrial': sub_df['Change Point'].values.astype(int),
        'Change Point Count': sub_df['Change Point Count'].values.astype(int),
    }
    df = pd.DataFrame(data)

    df_filtered = df.dropna(subset=['ChangepointTrial']).loc[df['ChangepointTrial'] != 0]
    plt.figure(figsize=(10, 8))
    sns.violinplot(x='Timepoint', y='ChangepointTrial', data=df_filtered,
                    palette=palette_violin)
    plt.title(f'Latest decreasing Changepoints Distribution by Age (months) — {chrom_label}')
    plt.ylabel('Changepoint')
    whole_trial_ticks()
    plt.show()

    pivot_table = df.pivot_table(
        index=regionType, columns='Timepoint',
        values='ChangepointTrial', aggfunc=np.mean,
    )
    pivot_table_no_zeros = pivot_table.replace(0, np.nan)
    pivot_table_no_zeros = pivot_table_no_zeros.sort_index()

    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table_no_zeros, cmap=paletteHeatmap,
                 annot=True, fmt='.0f', cbar=True)
    plt.title(f'Average Latest decreasing Changepoints Heatmap by {regionType} and Timepoint — {chrom_label}')
    plt.xlabel('Age')
    plt.ylabel(regionType)
    whole_trial_ticks()
    plt.show()

<!-- ## Data collation/summaries across tests -->

In [ ]:
# =============================================================================
# Summary of statistical test results across change-point groups × chromophore
# =============================================================================

RETAINED_TESTS = [
    'Decreasing',
    'First AND decreasing',
    'Final AND decreasing',
    'Earliest decreasing',
    'Latest decreasing',
]

if not statTestResults:
    print("No test results in statTestResults — did the individual test cells run?")
else:
    summary_rows = []
    for test_name in RETAINED_TESTS:
        for chrom_label in ['HbO', 'HbR']:
            key = (test_name, chrom_label)
            if key not in statTestResults:
                print(f"  WARNING: '{key}' not in statTestResults — skipping")
                continue
            res = statTestResults[key]
            summary_rows.append({
                'Test':               test_name,
                'Chromophore':        chrom_label,
                'n_obs':              res['n_obs'],
                'n_ages':             res['n_ages'],
                'ologit_coef':        res['ologit_coef'],
                'ologit_p':           res['ologit_pvalue'],
                'wald_stat':          res['wald_statistic'],
                'wald_p':             res['wald_pvalue'],
                'tukey_mean_diff':    res['tukey_meandiff'],
                'tukey_p':            res['tukey_pvalue'],
                'tukey_reject_H0':    res['tukey_reject'],
            })

    summaryDf = pd.DataFrame(summary_rows)

    def _sig_flag(row):
        if pd.isna(row['ologit_p']):
            return 'skipped'
        marks = []
        if row['ologit_p'] < 0.05:  marks.append('ologit')
        if row['wald_p']   < 0.05:  marks.append('Wald')
        if row['tukey_p']  < 0.05:  marks.append('Tukey')
        return ', '.join(marks) if marks else '—'
    summaryDf['sig_at_0.05'] = summaryDf.apply(_sig_flag, axis=1)

    # Print one table per chromophore
    for chrom_label in ['HbO', 'HbR']:
        print("=" * 110)
        print(f"SUMMARY — CHROMOPHORE: {chrom_label}")
        print("=" * 110)
        sub = summaryDf[summaryDf['Chromophore'] == chrom_label].copy()

        display_df = sub.copy()
        for col in ('ologit_coef', 'wald_stat', 'tukey_mean_diff'):
            display_df[col] = display_df[col].round(4)
        for col in ('ologit_p', 'wald_p', 'tukey_p'):
            display_df[col] = display_df[col].apply(
                lambda x: f"{x:.4g}" if not pd.isna(x) else 'NaN')
        print(display_df.drop(columns=['Chromophore']).to_string(index=False))
        print()
        n_sig = (sub['tukey_reject_H0'] == True).sum()
        print(f"  Tests with Tukey HSD rejecting H0 (uncorrected): "
              f"{n_sig}/{len(sub)}")
        print()

    # Save
    summaryDf.to_csv(
        projectRoot / 'data' / f'CHANGEPOINT_TEST_SUMMARY_by_chrom{preProcPipeline}{trialsIncluded}.csv',
        index=False,
    )

In [ ]:
# =============================================================================
# Multiple-comparison correction within pre-specified family, per chromophore
# =============================================================================
# 5 OLR tests corrected separately within each chromophore (HbO, HbR).

from statsmodels.stats.multitest import multipletests

FAMILY_TESTS = [
    'Decreasing',
    'First AND decreasing',
    'Final AND decreasing',
    'Earliest decreasing',
    'Latest decreasing',
]

correction_rows = []
for chrom_label in ['HbO', 'HbR']:
    fam_sub = summaryDf[
        (summaryDf['Test'].isin(FAMILY_TESTS)) &
        (summaryDf['Chromophore'] == chrom_label)
    ].copy()
    fam_sub = fam_sub.dropna(subset=['ologit_p'])
    fam_n = len(fam_sub)
    if fam_n == 0:
        print(f"\n{chrom_label}: no valid tests — skipping")
        continue

    print(f"\n{'=' * 100}")
    print(f"Habituation dynamics — {chrom_label}  (n = {fam_n} OLR tests corrected)")
    print(f"{'=' * 100}")

    raw_pvals = fam_sub['ologit_p'].astype(float).values
    if fam_n == 1:
        bonf_p = raw_pvals.copy()
        holm_p = raw_pvals.copy()
        bonf_reject = raw_pvals < 0.05
        holm_reject = raw_pvals < 0.05
    else:
        bonf_reject, bonf_p, _, _ = multipletests(raw_pvals, alpha=0.05,
                                                    method='bonferroni')
        holm_reject, holm_p, _, _ = multipletests(raw_pvals, alpha=0.05,
                                                    method='holm')

    for i, test_name in enumerate(fam_sub['Test'].values):
        correction_rows.append({
            'Chromophore':      chrom_label,
            'Test':             test_name,
            'ologit_coef':      float(fam_sub.iloc[i]['ologit_coef']),
            'ologit_raw_p':     float(raw_pvals[i]),
            'ologit_bonf_p':    float(bonf_p[i]),
            'ologit_holm_p':    float(holm_p[i]),
            'ologit_bonf_sig':  bool(bonf_reject[i]),
            'ologit_holm_sig':  bool(holm_reject[i]),
            'wald_stat':        float(fam_sub.iloc[i]['wald_stat']),
            'wald_raw_p':       float(fam_sub.iloc[i]['wald_p']),
            'tukey_mean_diff':  float(fam_sub.iloc[i]['tukey_mean_diff']),
            'tukey_raw_p':      float(fam_sub.iloc[i]['tukey_p']),
        })

correctionDf = pd.DataFrame(correction_rows)

# Format for display
display_corr = correctionDf.copy()
for col in ('ologit_coef', 'wald_stat', 'tukey_mean_diff'):
    display_corr[col] = display_corr[col].round(4)
for col in ('ologit_raw_p', 'ologit_bonf_p', 'ologit_holm_p',
            'wald_raw_p', 'tukey_raw_p'):
    display_corr[col] = display_corr[col].apply(lambda x: f"{x:.4g}")

for chrom_label in ['HbO', 'HbR']:
    print(f"\n{'=' * 100}")
    print(f"CORRECTED — {chrom_label}")
    print(f"{'=' * 100}")
    fam_rows = display_corr[display_corr['Chromophore'] == chrom_label]
    if fam_rows.empty:
        continue
    cols_to_show = [
        'Test', 'ologit_coef', 'ologit_raw_p', 'ologit_bonf_p', 'ologit_holm_p',
        'ologit_bonf_sig', 'ologit_holm_sig',
        'wald_stat', 'wald_raw_p',
        'tukey_mean_diff', 'tukey_raw_p',
    ]
    print(fam_rows[cols_to_show].to_string(index=False))

# Save
correctionDf.to_csv(
    projectRoot / 'data' / f'CHANGEPOINT_CORRECTION_by_chrom{preProcPipeline}{trialsIncluded}.csv',
    index=False,
)
print(f"\nWritten: CHANGEPOINT_CORRECTION_by_chrom{preProcPipeline}{trialsIncluded}.csv")

## ==== MESH PLOTTING ====

<!-- ## Mesh plotting -->

In [ ]:
# --- Age -> path key ---
ageKeyMap = {
    1:  '01mo',
    6:  '06mo',
    12: '12mo',
}

# --- Age-based colour palettes ---
age_colours = {
    1:  'Blues',
    6:  'Reds',
    12: 'Greens',
}

# --- Age-specific mesh + atlas paths ---
agePaths = {
    '01mo': {
        'atlas': dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '01mo' / '01mo_WangFunctional_Reg_Head.nii.gz',
        'mat':   dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface01mo.mat',
    },
    '06mo': {
        'atlas': dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '06mo' / '06mo_WangFunctional_Reg_Head.nii.gz',
        'mat':   dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface06mo.mat',
    },
    '12mo': {
        'atlas': dataDriveRoot / 'mri' / 'registered' / 'UNC_Surf_to_NeuroDev' / 'No Mask' / '12mo' / '12mo_WangFunctional_Reg_Head.nii.gz',
        'mat':   dataDriveRoot / 'mri' / 'meshes' / 'corticalSurface12mo.mat',
    },
}

In [ ]:
# =============================================================================
# Mesh plot — mean change point trial number per parcel, per age
# =============================================================================
# For the change-point group selected by `plotChangePointGroup`, compute the
# mean trial number of the change point per parcel, then plot on the age-
# appropriate cortical mesh.

# Map user-chosen group name to the corresponding all-ages dictionary
_GROUP_MAP = {
    'first':                all_firstChangePoints,
    'final':                all_lastChangePoints,
    'decreasing':           all_decrChangePoints,
    'first_and_decreasing': all_firstAndDecrChangePoints,
    'final_and_decreasing': all_finalAndDecrChangePoints,
    'earliest_decreasing':  all_earliestDecrChangePoints,
    'latest_decreasing':    all_latestDecrChangePoints,
}

if plotChangePointGroup not in _GROUP_MAP:
    raise ValueError(
        f"plotChangePointGroup '{plotChangePointGroup}' not recognised. "
        f"Choose from: {list(_GROUP_MAP.keys())}"
    )

plotDf = _GROUP_MAP[plotChangePointGroup]


def create_surface_plot_mean_cpt(age, vertices, faces, labels,
                                   parcel_means, title,
                                   cmin_val=None, cmax_val=None,
                                   colorscale='Viridis'):
    """
    Plot mean change point trial number per parcel on a 3D brain surface.
    parcel_means : dict mapping parcel_id -> mean change point trial number.
    Parcels with no data remain translucent grey.
    """
    # Map parcel means to vertices; NaN for unmapped parcels
    vertex_values = np.array([
        parcel_means.get(int(p), np.nan) for p in labels
    ])

    fig = go.Figure()

    # Mesh-driven figure size (as in create_surface_plot)
    extents = vertices.max(axis=0) - vertices.min(axis=0)
    max_extent = float(extents.max())
    ref_extent = 130.0
    ref_width  = 850
    width  = int(np.clip(ref_width * max_extent / ref_extent, 600, 1200))
    height = int(width * 0.85)

    # --- 1. Base anatomy (grey mesh) ---
    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color='lightgrey', opacity=0.15, showscale=False, hoverinfo='skip'
    ))

    # --- 2. Statistical data (coloured mesh) ---
    face_values = vertex_values[faces]
    significant_face_mask = np.any(~np.isnan(face_values), axis=1)
    sig_faces = faces[significant_face_mask]

    if len(sig_faces) > 0:
        # Replace NaNs with 0 for plotting; alpha is controlled by face mask
        intensity_plot = np.where(np.isnan(vertex_values), 0, vertex_values)

        if cmin_val is None:
            cmin_val = float(np.nanmin(vertex_values))
        if cmax_val is None:
            cmax_val = float(np.nanmax(vertex_values))

        fig.add_trace(go.Mesh3d(
            x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
            i=sig_faces[:, 0], j=sig_faces[:, 1], k=sig_faces[:, 2],
            intensity=intensity_plot,
            colorscale=colorscale,
            cmin=cmin_val,
            cmax=cmax_val,
            opacity=1.0,
            showscale=True,
            colorbar=dict(
                title='Mean CPt<br>trial number',
                thickness=15,
                len=0.7,
                x=0.95,
            ),
            lighting=dict(ambient=0.8)
        ))

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor='white',
            aspectmode='data',
            camera=dict(eye=dict(x=0.9, y=0.9, z=0.9))
        ),
        margin=dict(l=0, r=0, t=30, b=0),
        width=width, height=height,
    )
    return fig


# --- Compute shared colour scale bounds across ages so age plots are comparable ---
all_means = plotDf.groupby(['Age', 'Parcel'])['Change Point'].mean()
if all_means.empty:
    print(f"No change points in group '{plotChangePointGroup}' — nothing to plot.")
else:
    cmin_shared = float(all_means.min())
    cmax_shared = float(all_means.max())

    for age in uniqueAges:
        ageKey = ageKeyMap.get(age)
        if ageKey is None:
            print(f"  No mesh paths for age {age}, skipping")
            continue

        # Load mesh + atlas for this age
        paths = agePaths[ageKey]
        atlas_img = nib.load(paths['atlas'])
        atlas_data = atlas_img.get_fdata()
        mat = loadmat(paths['mat'])
        vertices = mat['vertices']
        faces = mat['faces'] - 1

        # Map vertices to parcel labels via atlas voxel space
        inv_affine = np.linalg.inv(atlas_img.affine)
        voxel_coords = nib.affines.apply_affine(inv_affine, vertices)
        voxel_coords = np.round(voxel_coords).astype(int)
        voxel_coords = np.clip(voxel_coords, [0, 0, 0], np.array(atlas_data.shape) - 1)
        labels = np.array([atlas_data[x, y, z] for x, y, z in voxel_coords])

        unlabelled_idx = np.where(labels == 0)[0]
        labelled_idx   = np.where(labels != 0)[0]
        if len(unlabelled_idx) > 0:
            tree = cKDTree(vertices[labelled_idx])
            _, nearest = tree.query(vertices[unlabelled_idx])
            labels[unlabelled_idx] = labels[labelled_idx][nearest]

        # Compute mean change point trial per parcel at this age
        age_sub = plotDf[plotDf['Age'] == age]
        parcel_means = (age_sub.groupby('Parcel')['Change Point']
                                .mean()
                                .to_dict())

        if not parcel_means:
            print(f"  No parcels with change points at age {age} — skipping plot")
            continue

        title = (f"Age {age}mo — mean trial number of "
                  f"'{plotChangePointGroup}' change points "
                  f"(n = {len(age_sub)} change points, "
                  f"n parcels = {len(parcel_means)})")

        fig = create_surface_plot_mean_cpt(
            age=age,
            vertices=vertices,
            faces=faces,
            labels=labels,
            parcel_means=parcel_means,
            title=title,
            cmin_val=cmin_shared,
            cmax_val=cmax_shared,
            colorscale=age_colours.get(age, 'Viridis'),  # your age-based palette
        )
        fig.show()